# Runtime binding lab

Binding is the setup phase before the first event runs. Use it when an object needs timeline-owned resources, especially deterministic RNG streams.


## 1. Imports and a tiny stochastic component

The component asks for its RNG stream in `bind(...)`, then uses that stream later during event handling.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

from simyuj.control import AgentContext, NodeAgent, SessionRuntime
from simyuj.engine import Component, Event, Timeline
from simyuj.network import Network, Node
from simyuj.runtime import BindingContext, bind_if_supported, bind_many


In [ ]:
class LossSampler(Component):
    def __init__(self, component_id: str, *, cutoff: float) -> None:
        self.component_id = component_id
        self.cutoff = cutoff
        self.rng = None
        self.bound_context = None
        self.samples = []

    def bind(self, context: BindingContext) -> None:
        self.bound_context = context
        self.rng = context.timeline.rng('loss-sampler', self.component_id)

    def handle_event(self, event, timeline) -> None:
        if self.rng is None:
            raise RuntimeError('bind this sampler before running events')
        sample = self.rng.random()
        self.samples.append((timeline.current_time, round(sample, 4), sample < self.cutoff))


## 2. The context is just the run context

The object being bound receives the active timeline, logger, optional IDs, and any small metadata the caller wants to attach.


In [ ]:
timeline = Timeline(master_seed=202)
context = BindingContext(
    timeline=timeline,
    logger=timeline.logger,
    entity_id='lab-session',
    component_id='fiber-a',
    meta=(('distance_km', 18), ('attenuation_db_per_km', 0.2)),
)

print('current time:', context.timeline.current_time)
print('component id:', context.component_id)
print('meta:', dict(context.meta))


## 3. Bind one object when it supports binding

Plain objects are skipped. Bind-capable objects get their setup phase.


In [ ]:
sampler = LossSampler('fiber-a', cutoff=0.45)
plain_object = object()

print('plain object bound:', bind_if_supported(plain_object, timeline))
print('sampler bound:', bind_if_supported(
    sampler,
    timeline,
    entity_id='lab-session',
    component_id='fiber-a',
    meta=(('role', 'quantum-link'),),
))
print('sampler has rng:', sampler.rng is not None)
print('sampler context meta:', dict(sampler.bound_context.meta))


In [ ]:
for tick in [10, 20, 30, 40]:
    timeline.schedule(Event(time=tick, target_ref=sampler, action='SAMPLE_LOSS', payload_ref=None))

timeline.run_until_empty()

print('samples:')
for item in sampler.samples:
    print(' t=', item[0], 'sample=', item[1], 'delivered=', item[2])
print('timeline stats:', timeline.stats)


## 4. `bind_many` keeps the setup pass boring and repeatable

Use it when you have a small collection and only some objects expose `bind(...)`.


In [ ]:
timeline = Timeline(master_seed=202)
samplers = [
    LossSampler('fiber-a', cutoff=0.45),
    object(),
    LossSampler('fiber-b', cutoff=0.70),
]

bound = bind_many(samplers, timeline)
print('bound count:', len(bound))
print('bound ids:', [item.component_id for item in bound])

for item in bound:
    timeline.schedule(Event(time=5, target_ref=item, action='SAMPLE_LOSS', payload_ref=None))

timeline.run_until_empty()
for item in bound:
    print(item.component_id, item.samples)


## 5. Late RNG setup is too late

The timeline freezes creation of new RNG streams when execution starts. Binding avoids surprises by creating streams before that point.


In [ ]:
class LateRng(Component):
    def handle_event(self, event, timeline) -> None:
        timeline.rng('late', 'stream')

late_timeline = Timeline(master_seed=9)
late_timeline.schedule(Event(time=1, target_ref=LateRng(), action='TRY_LATE_RNG', payload_ref=None))

try:
    late_timeline.run_one_step()
except RuntimeError as exc:
    print('late setup failed:', str(exc))


## 6. Networks bind devices once

A network can bind node devices and link transports before agents or events begin.


In [ ]:
class BindRecorder:
    def __init__(self, label: str) -> None:
        self.label = label
        self.contexts = []

    def bind(self, context: BindingContext) -> None:
        self.contexts.append(context)

network = Network('runtime-lab')
alice = Node('alice')
bob = Node('bob')
shared_device = BindRecorder('shared-memory')
bob_device = BindRecorder('bob-detector')

alice.add_device('memory', shared_device)
bob.add_device('detector', bob_device)
network.add_node(alice)
network.add_node(bob)

network_timeline = Timeline(master_seed=5)
bound_devices = network.bind_all(BindingContext(
    timeline=network_timeline,
    logger=network_timeline.logger,
    entity_id='network-lab',
))

print('bound devices:', [device.label for device in bound_devices])
for device in bound_devices:
    context = device.contexts[0]
    print(device.label, 'component=', context.component_id, 'meta=', dict(context.meta))


## 7. SessionRuntime runs the lifecycle for node agents

At the control layer, `SessionRuntime.run()` binds the network, binds agents, schedules starts, and drains the timeline.


In [ ]:
@dataclass(slots=True)
class Starter(NodeAgent):
    notes: list[str] = field(default_factory=list)

    def bind(self, context: BindingContext) -> None:
        self.notes.append(f'bound:{context.component_id}')

    def on_start(self, start, ctx: AgentContext) -> None:
        self.notes.append(f'started:{start.agent_id}:{ctx.node.node_id}:{ctx.session_id}')

runtime_timeline = Timeline(master_seed=77)
runtime_network = Network('session-lab')
node = Node('alice')
agent = Starter(agent_id='alice-agent', node_id='alice')
node.add_agent(agent)
runtime_network.add_node(node)

runtime = SessionRuntime(
    timeline=runtime_timeline,
    network=runtime_network,
    session_id='session-1',
)
runtime.run()

print('agent notes:', agent.notes)
print('timeline stats:', runtime_timeline.stats)


## Keep this model in your head

Bind first, then schedule and run. Binding is where objects declare timeline-owned resources; events are where the simulation behavior happens.
